In [1]:
import sys
!{sys.executable} -m pip install torch torch_geometric yfinance pandas numpy scikit-learn


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 65.1 MB/s eta 0:00:00


In [3]:
"""
GAT-Based Stock Portfolio Balancer with Transformer Temporal Encoding
======================================================================
Architecture:
  1. Transformer Encoder  — per-stock temporal embedding over a lookback window
  2. Graph Attention Net  — models inter-stock relationships
  3. Portfolio Head       — outputs a weight distribution (softmax)

Candle granularity is configurable: DAILY, WEEKLY, or MONTHLY.
Coarser candles reduce noise fed to the transformer while preserving
trend and momentum signal relevant for weekly rebalancing decisions.

Signal-to-noise trade-off summary:
  DAILY   — most data points, highest noise, suited for short-horizon signals
  WEEKLY  — aggregates 5 daily candles; smoother, better for weekly rebalance
  MONTHLY — very smooth, few candles per lookback, best for macro regime signals

Dependencies:
    pip install torch torch-geometric yfinance pandas numpy scikit-learn
"""

import math
import warnings
from datetime import datetime, timedelta
from enum import Enum

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv
from torch_geometric.utils import dense_to_sparse

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# 1. CANDLE FREQUENCY
# ---------------------------------------------------------------------------

class CandleFreq(Enum):
    """
    Granularity of OHLCV candles fed to the transformer encoder.

    Coarser candles = fewer sequence steps but higher SNR per step.
    The rebalance cadence (weekly) is independent of candle frequency —
    you always rebalance weekly; you just choose how smooth the input is.

    Recommended pairings
    ---------------------
    DAILY   -> lookback=60  candles  (~3 months of daily bars)
    WEEKLY  -> lookback=52  candles  (~1 year of weekly bars)   <- best default
    MONTHLY -> lookback=24  candles  (~2 years of monthly bars)
    """
    DAILY   = "D"
    WEEKLY  = "W"
    MONTHLY = "M"


# Sensible defaults per granularity
CANDLE_DEFAULTS = {
    CandleFreq.DAILY:   {"lookback": 60,  "history_years": 3},
    CandleFreq.WEEKLY:  {"lookback": 52,  "history_years": 10},
    CandleFreq.MONTHLY: {"lookback": 24,  "history_years": 6},
}

# ---------------------------------------------------------------------------
# 2. CONFIG  (change CANDLE_FREQ here to switch granularity)
# ---------------------------------------------------------------------------

CANDLE_FREQ         = CandleFreq.WEEKLY   # <-- main knob: DAILY / WEEKLY / MONTHLY

TICKERS             = ["AAPL", "MSFT", "GOOGL", "AMZN", "NVDA", "META", "TSLA", "JPM", "V", "UNH"]
LOOKBACK            = CANDLE_DEFAULTS[CANDLE_FREQ]["lookback"]
HISTORY_YEARS       = CANDLE_DEFAULTS[CANDLE_FREQ]["history_years"]
FEATURES_PER_BAR    = 5          # OHLCV
D_MODEL             = 64
N_HEADS_TRANSFORMER = 4
N_LAYERS_TRANSFORMER = 2
N_HEADS_GAT         = 4
GAT_LAYERS          = 2
DROPOUT             = 0.1
CORR_THRESHOLD      = 0.30
SEED                = 42

# Rebalance is always weekly; expressed in candle-units:
#   DAILY   -> every 5 candles  (5 days = 1 week)
#   WEEKLY  -> every 1 candle   (1 week = 1 week)
#   MONTHLY -> every 1 candle   (rebalance monthly — closest approximation)
REBALANCE_CANDLES = {
    CandleFreq.DAILY:   5,
    CandleFreq.WEEKLY:  1,
    CandleFreq.MONTHLY: 1,
}[CANDLE_FREQ]

torch.manual_seed(SEED)
np.random.seed(SEED)

# ---------------------------------------------------------------------------
# 3. DATA UTILITIES
# ---------------------------------------------------------------------------

def _resample_ohlcv(daily_df: pd.DataFrame, freq: CandleFreq) -> pd.DataFrame:
    """
    Resample a MultiIndex (price_col, ticker) daily DataFrame to target frequency
    using correct OHLCV aggregation rules.
    """
    if freq == CandleFreq.DAILY:
        return daily_df

    rule = "W-FRI" if freq == CandleFreq.WEEKLY else "ME"
    tickers = daily_df["Close"].columns.tolist()
    resampled = {}
    for t in tickers:
        ohlcv = daily_df.xs(t, axis=1, level=1)[["Open", "High", "Low", "Close", "Volume"]]
        rs = ohlcv.resample(rule).agg({
            "Open":   "first",
            "High":   "max",
            "Low":    "min",
            "Close":  "last",
            "Volume": "sum",
        }).dropna()
        for col in ["Open", "High", "Low", "Close", "Volume"]:
            resampled[(col, t)] = rs[col]

    result = pd.DataFrame(resampled)
    result.columns = pd.MultiIndex.from_tuples(result.columns)
    return result.sort_index()


def fetch_price_data(
    tickers: list,
    start: str,
    end: str,
    freq: CandleFreq = CANDLE_FREQ,
) -> pd.DataFrame:
    """
    Download daily OHLCV then resample to `freq`.
    Falls back to synthetic data if yfinance is unavailable.
    """
    try:
        import yfinance as yf
        raw = yf.download(tickers, start=start, end=end, auto_adjust=True, progress=False)
        if not isinstance(raw.columns, pd.MultiIndex):
            raw = pd.concat({tickers[0]: raw}, axis=1).swaplevel(axis=1)
    except Exception as exc:
        print(f"[WARN] yfinance failed ({exc}). Using synthetic data.")
        raw = _synthetic_data(tickers, start, end)

    df = _resample_ohlcv(raw, freq)
    print(f"Candle freq : {freq.name} | bars: {len(df)} | "
          f"lookback: {LOOKBACK} candles | "
          f"rebalance every: {REBALANCE_CANDLES} candle(s)")
    return df


def _synthetic_data(tickers, start, end) -> pd.DataFrame:
    dates = pd.bdate_range(start, end)
    rng   = np.random.default_rng(SEED)
    n     = len(dates)
    price = 100 * np.cumprod(1 + rng.normal(0.0005, 0.015, (n, len(tickers))), axis=0)
    data  = {}
    for i, t in enumerate(tickers):
        data[("Open",   t)] = price[:, i] * (1 - rng.uniform(0, 0.005, n))
        data[("High",   t)] = price[:, i] * (1 + rng.uniform(0, 0.010, n))
        data[("Low",    t)] = price[:, i] * (1 - rng.uniform(0, 0.010, n))
        data[("Close",  t)] = price[:, i]
        data[("Volume", t)] = rng.integers(1_000_000, 10_000_000, n).astype(float)
    return pd.DataFrame(data, index=dates)


def build_feature_tensor(
    df: pd.DataFrame,
    tickers: list,
    end_idx: int,
    lookback: int,
) -> torch.Tensor:
    """
    Returns (n_stocks, lookback, 5) with min-max normalised OHLCV.
    Works identically regardless of candle frequency.
    """
    window = df.iloc[end_idx - lookback : end_idx]
    feat_list = []
    for t in tickers:
        ohlcv = window.xs(t, level=1, axis=1)[["Open", "High", "Low", "Close", "Volume"]].values.astype(np.float32)
        #ohlcv = window[["Open", "High", "Low", "Close", "Volume"]][t].values.astype(np.float32)
        col_min = ohlcv.min(axis=0, keepdims=True)
        col_max = ohlcv.max(axis=0, keepdims=True)
        ohlcv = (ohlcv - col_min) / (col_max - col_min + 1e-8)
        feat_list.append(ohlcv)
    return torch.tensor(np.stack(feat_list), dtype=torch.float32)  # (S, T, F)


def build_correlation_graph(df, tickers, end_idx, lookback, threshold=CORR_THRESHOLD):
    """Correlation-based graph over the lookback window (candle-freq agnostic)."""
    returns = df["Close"][tickers].iloc[end_idx - lookback : end_idx].pct_change().dropna()
    corr    = returns.corr().values.astype(np.float32)
    adj     = torch.tensor(np.abs(corr) > threshold, dtype=torch.float32)
    return dense_to_sparse(adj)


# ---------------------------------------------------------------------------
# 4. TRANSFORMER TEMPORAL ENCODER
# ---------------------------------------------------------------------------

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 512, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return self.dropout(x + self.pe[:, : x.size(1)])


class StockTemporalEncoder(nn.Module):
    """
    Shared-weight transformer: OHLCV sequence -> fixed-size embedding.
    Sequence length = `lookback` candles at the chosen frequency.
    With WEEKLY candles the transformer attends over ~1 year of weekly bars
    instead of 60 noisy daily bars, giving it much cleaner trend signal.
    """
    def __init__(self, n_features=FEATURES_PER_BAR, d_model=D_MODEL,
                 n_heads=N_HEADS_TRANSFORMER, n_layers=N_LAYERS_TRANSFORMER,
                 dropout=DROPOUT):
        super().__init__()
        self.input_proj = nn.Linear(n_features, d_model)
        self.pos_enc    = PositionalEncoding(d_model, dropout=dropout)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.pool = nn.Linear(d_model, d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):              # x: (S, T, F)
        h = self.pos_enc(self.input_proj(x))
        h = self.transformer(h)
        h = h.mean(dim=1)
        return self.norm(F.gelu(self.pool(h)))


# ---------------------------------------------------------------------------
# 5. GAT PORTFOLIO NETWORK
# ---------------------------------------------------------------------------

class GATPortfolioNet(nn.Module):
    def __init__(self, n_stocks, n_features=FEATURES_PER_BAR, d_model=D_MODEL,
                 n_heads_tf=N_HEADS_TRANSFORMER, n_layers_tf=N_LAYERS_TRANSFORMER,
                 n_heads_gat=N_HEADS_GAT, gat_layers=GAT_LAYERS, dropout=DROPOUT):
        super().__init__()
        self.n_stocks = n_stocks
        self.temporal_encoder = StockTemporalEncoder(
            n_features, d_model, n_heads_tf, n_layers_tf, dropout)
        self.temperature = nn.Parameter(torch.tensor(1.0))

        self.gat_convs = nn.ModuleList()
        in_dim = d_model
        for _ in range(gat_layers):
            self.gat_convs.append(
                GATConv(in_dim, d_model // n_heads_gat,
                        heads=n_heads_gat, dropout=dropout, concat=True))
            in_dim = d_model

        self.gat_norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, 1),
        )

    def forward(self, x, edge_index, edge_attr):
        node_emb = self.temporal_encoder(x)          # (S, D)
        h = node_emb
        for conv in self.gat_convs:
            h = F.elu(conv(h, edge_index))
        h = self.gat_norm(h + node_emb)
        scores  = self.head(h).squeeze(-1)
        weights = F.softmax(scores / self.temperature.clamp(min=0.1), dim=0)
        return weights


# ---------------------------------------------------------------------------
# 6. PORTFOLIO LOSS
# ---------------------------------------------------------------------------

class PortfolioEnv:
    ##@staticmethod
    #def sharpe_loss(weights, future_returns, risk_free=0.0):
    #    port_ret = (weights * future_returns).sum()
    #    port_std = torch.sqrt(
    #        (weights ** 2 * (future_returns - future_returns.mean()) ** 2).sum() + 1e-8)
    #    return -(port_ret - risk_free) / port_std
#

    @staticmethod
    def sharpe_loss(weights, future_returns, risk_free=0.0):
        port_ret = (weights * future_returns).sum()
        port_std = torch.sqrt(
            (weights ** 2 * (future_returns - future_returns.mean()) ** 2).sum() + 1e-8
        )
        sharpe = (port_ret - risk_free) / port_std

        # Penalise entropy (pushes away from uniform) — tune lambda 0.05–0.2
        entropy = -(weights * (weights + 1e-8).log()).sum()
        entropy_penalty = 0.1 * entropy

        return -sharpe + entropy_penalty

    @staticmethod
    def compute_forward_returns(df, tickers, idx, horizon):
        """Returns over next `horizon` candles — works for any candle freq."""
        close = df["Close"][tickers]
        cur   = close.iloc[idx].values
        fut   = close.iloc[min(idx + horizon, len(close) - 1)].values
        return torch.tensor((fut - cur) / (cur + 1e-8), dtype=torch.float32)


# ---------------------------------------------------------------------------
# 7. TRAINING LOOP
# ---------------------------------------------------------------------------

def train(model, df, tickers, lookback=LOOKBACK, epochs=100,
          rebalance_freq=REBALANCE_CANDLES, lr=3e-4, device="cpu"):
    model  = model.to(device)
    optim  = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched  = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=epochs)
    n_bars = len(df)
    steps  = list(range(lookback, n_bars - rebalance_freq, rebalance_freq))

    print(f"\nTraining | {CANDLE_FREQ.name} candles | "
          f"{len(steps)} rebalance steps x {epochs} epochs")
    print(f"Stocks : {tickers}\n{'─'*60}")

    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss = 0.0
        for idx in steps:
            x       = build_feature_tensor(df, tickers, idx, lookback).to(device)
            ei, ea  = build_correlation_graph(df, tickers, idx, lookback)
            ei, ea  = ei.to(device), ea.to(device)
            weights = model(x, ei, ea)
            fut_ret = PortfolioEnv.compute_forward_returns(
                          df, tickers, idx, rebalance_freq).to(device)
            loss    = PortfolioEnv.sharpe_loss(weights, fut_ret)
            optim.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optim.step()
            epoch_loss += loss.item()

        sched.step()
        if epoch % 5 == 0 or epoch == 1:
            avg = epoch_loss / max(len(steps), 1)
            print(f"Epoch {epoch:3d}/{epochs} | avg -Sharpe: {avg:.4f} | "
                  f"lr: {sched.get_last_lr()[0]:.2e}")
    return model


# ---------------------------------------------------------------------------
# 8. INFERENCE
# ---------------------------------------------------------------------------

def rebalance(model, df, tickers, lookback=LOOKBACK, device="cpu") -> pd.Series:
    """Run model on the most recent window, return target portfolio weights."""
    model.eval()
    model.to(device)
    end_idx = len(df)
    with torch.no_grad():
        x      = build_feature_tensor(df, tickers, end_idx, lookback).to(device)
        ei, ea = build_correlation_graph(df, tickers, end_idx, lookback)
        ei, ea = ei.to(device), ea.to(device)
        w      = model(x, ei, ea).cpu().numpy()
    return pd.Series(w, index=tickers).sort_values(ascending=False)


# ---------------------------------------------------------------------------
# 9. BACKTEST
# ---------------------------------------------------------------------------

def backtest(model, df, tickers, lookback=LOOKBACK,
             rebalance_freq=REBALANCE_CANDLES, device="cpu") -> pd.DataFrame:
    model.eval()
    records = []
    n_bars  = len(df)
    periods_per_year = {
        CandleFreq.DAILY:   252 / rebalance_freq,
        CandleFreq.WEEKLY:  52  / rebalance_freq,
        CandleFreq.MONTHLY: 12  / rebalance_freq,
    }[CANDLE_FREQ]

    for idx in range(lookback, n_bars - rebalance_freq, rebalance_freq):
        x      = build_feature_tensor(df, tickers, idx, lookback).to(device)
        ei, ea = build_correlation_graph(df, tickers, idx, lookback)
        ei, ea = ei.to(device), ea.to(device)
        with torch.no_grad():
            w = model(x, ei, ea).cpu().numpy()
        fut = (df["Close"][tickers].iloc[idx + rebalance_freq].values /
               df["Close"][tickers].iloc[idx].values - 1)
        records.append({
            "date": df.index[idx],
            "portfolio_ret": float((w * fut).sum()),
            **{t: float(wi) for t, wi in zip(tickers, w)},
        })

    result = pd.DataFrame(records).set_index("date")
    result["cumulative_ret"] = (1 + result["portfolio_ret"]).cumprod() - 1

    r   = result["portfolio_ret"]
    ann = (1 + r.mean()) ** periods_per_year - 1
    sh  = r.mean() / (r.std() + 1e-8) * math.sqrt(periods_per_year)
    mdd = ((result["cumulative_ret"] + 1).cummax() -
           (result["cumulative_ret"] + 1)).max()

    print(f"\nBacktest  [{CANDLE_FREQ.name} candles]")
    print(f"  Total return      : {result['cumulative_ret'].iloc[-1]*100:.2f}%")
    print(f"  Annualised return : {ann*100:.2f}%")
    print(f"  Sharpe            : {sh:.3f}")
    print(f"  Max drawdown      : {mdd*100:.2f}%")
    return result


# ---------------------------------------------------------------------------
# 10. MAIN
# ---------------------------------------------------------------------------

def main():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device      : {device}")
    print(f"Candle freq : {CANDLE_FREQ.name}  "
          f"(lookback={LOOKBACK}, rebalance_every={REBALANCE_CANDLES} candle(s))")

    end_date   = datetime.today().strftime("%Y-%m-%d")
    start_date = (datetime.today() - timedelta(days=HISTORY_YEARS * 365)).strftime("%Y-%m-%d")
    print(f"Data range  : {start_date} -> {end_date}")

    df = fetch_price_data(TICKERS, start_date, end_date, freq=CANDLE_FREQ)
    df = df.ffill().dropna()
    print(f"Bars after resample & dropna: {len(df)}")

    split    = int(len(df) * 0.75)
    train_df = df.iloc[:split]

    model = GATPortfolioNet(n_stocks=len(TICKERS))
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Model parameters: {total_params:,}\n")

    model = train(model, train_df, TICKERS, device=device)

    print("\n" + "─" * 60)
    result = backtest(model, df, TICKERS, device=device)

    print("\nRecommended weights (next rebalance):")
    weights = rebalance(model, df, TICKERS, device=device)
    print(weights.to_string(float_format=lambda x: f"{x:.4f}"))
    print(f"\nSum of weights: {weights.sum():.6f}")

    return model, result, weights


if __name__ == "__main__":
    model, backtest_results, weights = main()

Device      : cuda
Candle freq : WEEKLY  (lookback=52, rebalance_every=1 candle(s))
Data range  : 2016-03-21 -> 2026-03-19
Candle freq : WEEKLY | bars: 522 | lookback: 52 candles | rebalance every: 1 candle(s)
Bars after resample & dropna: 522
Model parameters: 115,458


Training | WEEKLY candles | 338 rebalance steps x 100 epochs
Stocks : ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META', 'TSLA', 'JPM', 'V', 'UNH']
────────────────────────────────────────────────────────────
Epoch   1/100 | avg -Sharpe: -0.3349 | lr: 3.00e-04
Epoch   5/100 | avg -Sharpe: -0.3374 | lr: 2.98e-04
Epoch  10/100 | avg -Sharpe: -0.3198 | lr: 2.93e-04
Epoch  15/100 | avg -Sharpe: -0.3429 | lr: 2.84e-04
Epoch  20/100 | avg -Sharpe: -0.3461 | lr: 2.71e-04
Epoch  25/100 | avg -Sharpe: -0.3544 | lr: 2.56e-04
Epoch  30/100 | avg -Sharpe: -0.3220 | lr: 2.38e-04
Epoch  35/100 | avg -Sharpe: -0.3867 | lr: 2.18e-04
Epoch  40/100 | avg -Sharpe: -0.4063 | lr: 1.96e-04
Epoch  45/100 | avg -Sharpe: -0.7512 | lr: 1.73e-04


In [ ]:
# ---------------------------------------------------------------------------
# BACKTEST: EQUAL-WEIGHT BUY & HOLD (no rebalancing)
# ---------------------------------------------------------------------------

def backtest_equal_weight(
    df: pd.DataFrame,
    tickers: list,
    lookback: int = LOOKBACK,
) -> pd.DataFrame:
    """
    Starts at the same index the model backtest starts (after the lookback
    warm-up), allocates 1/N to each stock, and never rebalances.
    Tracks daily P&L based on how each position drifts from its initial weight.
    """
    start_idx  = lookback
    n          = len(tickers)
    init_close = df["Close"][tickers].iloc[start_idx].values          # entry prices
    init_w     = np.ones(n) / n                                        # 1/N weights
    # Number of "shares" per unit of capital
    shares     = init_w / init_close                                   # (n,)

    close_matrix = df["Close"][tickers].iloc[start_idx:].values       # (days, n)

    # Portfolio value each day (normalised so day-0 = 1.0)
    port_values  = (close_matrix * shares).sum(axis=1)
    port_values  = port_values / port_values[0]

    dates  = df.index[start_idx:]
    result = pd.DataFrame({
        "portfolio_value": port_values,
        "daily_ret":       pd.Series(port_values).pct_change().fillna(0).values,
    }, index=dates)

    # Drifted weights over time (informational)
    total_val = close_matrix * shares                                  # (days, n)
    for i, t in enumerate(tickers):
        result[f"w_{t}"] = total_val[:, i] / total_val.sum(axis=1)

    # ── Stats ────────────────────────────────────────────────────────────────
    periods_per_year = {
        CandleFreq.DAILY:   252,
        CandleFreq.WEEKLY:  52,
        CandleFreq.MONTHLY: 12,
    }[CANDLE_FREQ]

    r        = result["daily_ret"]
    ann_ret  = result["portfolio_value"].iloc[-1] ** (periods_per_year / len(result)) - 1
    sharpe   = r.mean() / (r.std() + 1e-8) * math.sqrt(periods_per_year)
    roll_max = result["portfolio_value"].cummax()
    mdd      = ((roll_max - result["portfolio_value"]) / roll_max).max()

    print(f"\nEqual-Weight Buy & Hold  [{CANDLE_FREQ.name} candles, 1/{n} each, no rebalancing]")
    print(f"  Total return      : {(result['portfolio_value'].iloc[-1]-1)*100:.2f}%")
    print(f"  Annualised return : {ann_ret*100:.2f}%")
    print(f"  Sharpe            : {sharpe:.3f}")
    print(f"  Max drawdown      : {mdd*100:.2f}%")

    # Show how much weights drifted by end of period
    start_w = {t: 1/n for t in tickers}
    end_w   = {t: result[f"w_{t}"].iloc[-1] for t in tickers}
    print(f"\n  Weight drift (start -> end):")
    for t in tickers:
        print(f"    {t:<6}  {start_w[t]*100:.1f}%  ->  {end_w[t]*100:.1f}%")

    return result

In [ ]:
def main():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device      : {device}")
    print(f"Candle freq : {CANDLE_FREQ.name}  "
          f"(lookback={LOOKBACK}, rebalance_every={REBALANCE_CANDLES} candle(s))")

    end_date   = datetime.today().strftime("%Y-%m-%d")
    start_date = (datetime.today() - timedelta(days=HISTORY_YEARS * 365)).strftime("%Y-%m-%d")
    print(f"Data range  : {start_date} -> {end_date}")

    df = fetch_price_data(TICKERS, start_date, end_date, freq=CANDLE_FREQ)
    df = df.ffill().dropna()
    print(f"Bars after resample & dropna: {len(df)}")


    ew_result = backtest_equal_weight(df, TICKERS)




    print("\n" + "─" * 60)
    result = backtest(model, df, TICKERS, device=device)

    print("\nRecommended weights (next rebalance):")
    weights = rebalance(model, df, TICKERS, device=device)
    print(weights.to_string(float_format=lambda x: f"{x:.4f}"))
    print(f"\nSum of weights: {weights.sum():.6f}")

    return model, result, weights


if __name__ == "__main__":
    model, backtest_results, weights = main()
# in main(), after the existing backtest() call:


Device      : cuda
Candle freq : WEEKLY  (lookback=52, rebalance_every=1 candle(s))
Data range  : 2021-03-19 -> 2026-03-18
Candle freq : WEEKLY | bars: 262 | lookback: 52 candles | rebalance every: 1 candle(s)
Bars after resample & dropna: 262

Equal-Weight Buy & Hold  [WEEKLY candles, 1/10 each, no rebalancing]
  Total return      : 120.39%
  Annualised return : 21.61%
  Sharpe            : 0.962
  Max drawdown      : 30.20%

  Weight drift (start -> end):
    AAPL    10.0%  ->  7.2%
    MSFT    10.0%  ->  6.2%
    GOOGL   10.0%  ->  10.4%
    AMZN    10.0%  ->  6.1%
    NVDA    10.0%  ->  31.3%
    META    10.0%  ->  13.2%
    TSLA    10.0%  ->  6.0%
    JPM     10.0%  ->  10.3%
    V       10.0%  ->  6.6%
    UNH     10.0%  ->  2.8%

────────────────────────────────────────────────────────────

Backtest  [WEEKLY candles]
  Total return      : 114.08%
  Annualised return : 23.88%
  Sharpe            : 0.957
  Max drawdown      : 42.92%

Recommended weights (next rebalance):
UNH     0